# Brown bear habitat suitability modelling

This notebook builds data-conservative habitat models for female brown bears with cubs younger than six months and between six and twelve months. It compares full-Pyrenees and local availability definitions using shared evaluation backgrounds, selects one robust final candidate per cub class, and creates exactly one official suitability map and one official zone map for each class.

Analytical rasters remain in EPSG:3035 for correct 100 m distances and areas. Presentation maps are reprojected to EPSG:4326 so the Pyrenean bounding box is geographically intuitive.

## 0. Run controls

Every execution receives its own `run_N` output directory from the launcher. `SMOKE_MODE=True` reduces backgrounds, tuning trials, bootstrap repetitions, and disables raster prediction so structural changes can be tested quickly. A complete scientific run must use `SMOKE_MODE=False`.

In [25]:
OUTPUT_DIR = "outputs/ursus_arctos_habitat_modelling/run_2"
RANDOM_SEED = 42
SMOKE_MODE = False

N_BACKGROUND_TRAIN = 10_000
N_BACKGROUND_TEST = 2_000
MIN_THIN_DISTANCE_M = 100
N_THIN_ITERATIONS = 20
LOCAL_BUFFER_KM = 25.0
LOCAL_BUFFER_SENSITIVITY_KM = (20.0, 25.0, 30.0)
CUSTOM_LOCAL_DOMAIN_PATH = None

N_FINAL_BOOTSTRAP_REPLICATES = 10
N_EVALUATION_REPLICATES = 5
N_PAPER_VALIDATION_REPEATS = 10
RUN_TUNING = True
STRICT_NESTED_TUNING = True
N_RF_TUNING_ITER = 12
N_OPTUNA_TRIALS = 20
INNER_BLOCKS_AXIS = 2
RUN_XGBOOST = True
RUN_SHAP = True
RUN_MAP_PREDICTION = True
BATCH_SIZE = 100_000


## 0.1 Environment and run setup

This cell locates the repository, loads the reusable workflow module, materializes the run configuration, creates run-owned output folders, and starts a dedicated log. Expected output: the resolved project and run directories.

In [26]:
import os
import sys
from pathlib import Path

def find_project_root(start: Path) -> Path:
    for candidate in [start.resolve(), *start.resolve().parents]:
        if (candidate / "pyproject.toml").exists() and (candidate / "configs").exists():
            return candidate
    raise FileNotFoundError("Could not find the pirineus-raster project root.")

PROJECT_ROOT = find_project_root(Path.cwd())
os.chdir(PROJECT_ROOT)
sys.path.insert(0, str(PROJECT_ROOT / "notebooks" / "ursus_arctos_project"))

from habitat_modelling_workflow import HabitatWorkflow, WorkflowConfig

config = WorkflowConfig(
    output_dir=OUTPUT_DIR,
    random_seed=RANDOM_SEED,
    smoke_mode=SMOKE_MODE,
    n_background_train=N_BACKGROUND_TRAIN,
    n_background_test=N_BACKGROUND_TEST,
    min_thin_distance_m=MIN_THIN_DISTANCE_M,
    n_thin_iterations=N_THIN_ITERATIONS,
    local_buffer_km=LOCAL_BUFFER_KM,
    local_buffer_sensitivity_km=tuple(LOCAL_BUFFER_SENSITIVITY_KM),
    custom_local_domain_path=CUSTOM_LOCAL_DOMAIN_PATH,
    n_final_bootstrap_replicates=N_FINAL_BOOTSTRAP_REPLICATES,
    n_evaluation_replicates=N_EVALUATION_REPLICATES,
    n_paper_validation_repeats=N_PAPER_VALIDATION_REPEATS,
    run_tuning=RUN_TUNING,
    strict_nested_tuning=STRICT_NESTED_TUNING,
    n_rf_tuning_iter=N_RF_TUNING_ITER,
    n_optuna_trials=N_OPTUNA_TRIALS,
    inner_blocks_axis=INNER_BLOCKS_AXIS,
    run_xgboost=RUN_XGBOOST,
    run_shap=RUN_SHAP,
    run_map_prediction=RUN_MAP_PREDICTION,
    batch_size=BATCH_SIZE,
)
workflow = HabitatWorkflow(config, PROJECT_ROOT)
print(f"Project root: {workflow.project_root}")
print(f"Run root: {workflow.paths.root}")


2026-06-12 18:50:07,375 | INFO | Project root: /mnt/csl/work/aniol.garriga.torra/pirineus_raster
2026-06-12 18:50:07,391 | INFO | Run root: /mnt/csl/work/aniol.garriga.torra/pirineus_raster/outputs/ursus_arctos_habitat_modelling/run_2


Project root: /mnt/csl/work/aniol.garriga.torra/pirineus_raster
Run root: /mnt/csl/work/aniol.garriga.torra/pirineus_raster/outputs/ursus_arctos_habitat_modelling/run_2


# 1. Data: inspection and treatment

This stage inventories every environmental variable by variable name, verifies raster alignment, profiles missingness and value ranges, and builds the common complete-predictor mask. It then cleans GPS and OBS data, explicitly parses mixed European and ISO date formats, assigns the environmental season from each event date, maps points to the official 100 m grid, and performs the existing grid-scale spatial treatment.

No additional destructive temporal thinning is applied. Instead, each bear-day becomes a trajectory group used later for weighting and bootstrap uncertainty. Expected output: environmental quality tables, cleaning reports with zero recoverable OBS date losses, and presence counts by bear, class, season, and day.

In [27]:
data_outputs = workflow.inspect_and_treat_data()
display(data_outputs["gps_cleaning"])
display(data_outputs["obs_cleaning"])
display(data_outputs["environmental_quality"].sort_values("valid_pct").head(20))
display(data_outputs["presence_summary"])


2026-06-12 18:50:22,380 | INFO | All target OBS records have parseable dates and seasons.


,step,before,after,removed
0,drop duplicated id_obs,4629,4629,0
1,keep target cub classes,4629,4629,0
2,keep points inside AOI lon/lat bounds,4629,4629,0
3,keep points with parseable event season,4629,4628,1
4,remove records after mortality/disappearance year,4628,4628,0


,step,before,after,removed
0,drop duplicated id_obs,1962,1962,0
1,keep target cub classes,1962,1190,772
2,keep points inside AOI lon/lat bounds,1190,1167,23
3,keep points with parseable event season,1167,1167,0


,variable,model_variable,temporal_role,temporal_dimension,valid_count,total_pixels,valid_pct,nodata_pct,min,max,mean,std,zero_pct,issue,path
0,climate_max_temperature_autumn,climate_max_temperature,seasonal,autumn,4840678,5456000,88.722104,11.277896,2.361523,22.249125,16.127676,3.465143,0.000000,valid_coverage_below_90pct,/mnt/csl/work/aniol.garriga.torra/pirineus_ras...
1,climate_max_temperature_spring,climate_max_temperature,seasonal,spring,4840678,5456000,88.722104,11.277896,-1.583383,21.508011,14.106711,4.128107,0.000000,valid_coverage_below_90pct,/mnt/csl/work/aniol.garriga.torra/pirineus_ras...
2,climate_max_temperature_summer,climate_max_temperature,seasonal,summer,4840678,5456000,88.722104,11.277896,6.755135,32.950741,23.793464,4.505514,0.000000,valid_coverage_below_90pct,/mnt/csl/work/aniol.garriga.torra/pirineus_ras...
3,climate_max_temperature_winter,climate_max_temperature,seasonal,winter,4840678,5456000,88.722104,11.277896,-4.236337,14.763221,7.906586,2.868097,0.000000,valid_coverage_below_90pct,/mnt/csl/work/aniol.garriga.torra/pirineus_ras...
4,climate_mean_temperature_autumn,climate_mean_temperature,seasonal,autumn,4840678,5456000,88.722104,11.277896,-1.254287,16.374548,10.736450,3.099612,0.000000,valid_coverage_below_90pct,/mnt/csl/work/aniol.garriga.torra/pirineus_ras...
5,climate_mean_temperature_spring,climate_mean_temperature,seasonal,spring,4840678,5456000,88.722104,11.277896,-5.008668,14.528477,8.589264,3.543675,0.000000,valid_coverage_below_90pct,/mnt/csl/work/aniol.garriga.torra/pirineus_ras...
6,climate_mean_temperature_summer,climate_mean_temperature,seasonal,summer,4840678,5456000,88.722104,11.277896,3.784811,24.635437,17.492411,3.724220,0.000000,valid_coverage_below_90pct,/mnt/csl/work/aniol.garriga.torra/pirineus_ras...
7,climate_mean_temperature_winter,climate_mean_temperature,seasonal,winter,4840678,5456000,88.722104,11.277896,-7.703032,9.086433,3.150163,2.757902,0.000000,valid_coverage_below_90pct,/mnt/csl/work/aniol.garriga.torra/pirineus_ras...
8,climate_min_temperature_autumn,climate_min_temperature,seasonal,autumn,4840678,5456000,88.722104,11.277896,-5.330054,11.295947,5.447716,2.778843,0.000475,valid_coverage_below_90pct,/mnt/csl/work/aniol.garriga.torra/pirineus_ras...
9,climate_min_temperature_spring,climate_min_temperature,seasonal,spring,4840678,5456000,88.722104,11.277896,-8.765359,8.662493,3.112548,3.043069,0.000103,valid_coverage_below_90pct,/mnt/csl/work/aniol.garriga.torra/pirineus_ras...


,cub_class,bear_name,event_season,n_presences,date_min,date_max,n_days
0,6to12,Hvala,autumn,22,2007-09-14 11:10:00+00:00,2009-11-03 00:00:00+00:00,21
1,6to12,Hvala,spring,3,2009-04-29 00:00:00+00:00,2009-05-30 00:00:00+00:00,3
2,6to12,Hvala,summer,53,2007-07-01 00:01:00+00:00,2009-08-26 00:00:00+00:00,42
3,6to12,Melba,autumn,5,1997-09-02 07:00:00+00:00,1997-09-27 09:27:00+00:00,5
4,6to12,Melba,summer,45,1997-07-01 07:01:00+00:00,1997-08-31 09:30:00+00:00,44
5,6to12,Sorita,autumn,226,2019-09-01 00:03:00+00:00,2019-11-16 16:20:00+00:00,71
6,6to12,Sorita,summer,389,2019-07-01 04:00:00+00:00,2019-08-31 22:00:00+00:00,61
7,6to12,Ziva,autumn,72,1997-09-02 09:30:00+00:00,1997-11-23 09:50:00+00:00,72
8,6to12,Ziva,summer,52,1997-07-01 07:00:00+00:00,1997-08-31 07:00:00+00:00,52
9,lt6,Hvala,spring,66,2007-05-01 03:01:00+00:00,2007-05-29 18:01:00+00:00,22


## 1.1 Domains, background isolation, and model tables

The full domain uses the complete project grid. The primary local domain is the union of separate movement hulls for each bear buffered by 25 km, preserving the western and central ranges as separate components instead of joining them with one convex hull. The notebook also reports 20, 25, and 30 km sensitivity areas.

Four deterministic and non-overlapping background pools are reserved before fitting: local training, full training, shared local testing, and shared full testing. Seasonal predictors for each background pool follow the observed class-specific seasonal distribution. Expected output: domain tables and EPSG:4326 domain plots, isolated background CSVs, and complete model feature tables.

In [28]:
domain_outputs = workflow.prepare_domains_and_model_data()
display(domain_outputs["domain_summary"])
display(domain_outputs["domain_sensitivity"])
display(domain_outputs["feature_set"])


,domain,area_km2,valid_area_km2,n_components
0,full_aoi,54560.000000,46795.90,1
1,local_domain,10775.934906,10377.98,2


,buffer_km,area_km2,n_components,contains_all_gps,selected,source
0,20.0,8198.001143,2,True,False,separate_bear_hulls
1,25.0,10775.934906,2,True,True,separate_bear_hulls
2,30.0,13640.579868,1,True,False,separate_bear_hulls


,feature,in_reduced_set
0,climate_max_temperature,False
1,climate_mean_temperature,True
2,climate_min_temperature,False
3,climate_precipitation_annual,True
4,climate_snow_cover,True
5,climate_solar_radiation_annual,True
6,climate_water_availability,True
7,habitat_biomass_mean,True
8,habitat_broadleaf,True
9,habitat_coniferous_dominant_leaf_type,True


# 2. Modelling and model selection

LOIO remains the primary evaluation: one bear is withheld while all other bears are used for fitting. Presence observations are not discarded to balance individuals. Sample weights instead give equal total influence to bears and to observed days within each bear. Hyperparameters are tuned inside each outer fold when strict nested tuning is enabled.

Every full-domain and local-domain RF, XGBoost, and weighted RF/XGBoost blend is evaluated on the same held-out bear and the same local and full background cells. Selection first maximizes the weakest of local mean AUC, full mean AUC, and worst-bear local AUC; average AUC, Boyce, and Brier break ties. The paper-style repeated random 75/25 result is reported only for comparison because it mixes locations from the same bears across train and test. Expected output: fold metrics, equal-bear summaries, selected models, calibration diagnostics, and fitted daily-block bootstrap ensembles.

In [29]:
model_outputs = workflow.evaluate_and_select_models()
display(model_outputs["winners"])
display(model_outputs["loio_summary"].sort_values(["cub_class", "evaluation_scenario", "auc_roc_mean"], ascending=[True, True, False]))
display(model_outputs["paper_validation"].groupby("cub_class")[["auc_roc", "boyce", "brier"]].agg(["mean", "std"]))
display(model_outputs["calibration"])


2026-06-12 18:51:40,093 | INFO | full_aoi_lt6_rf_Hvala RF candidate 1/12 AUC=0.884
2026-06-12 18:51:51,321 | INFO | full_aoi_lt6_rf_Hvala RF candidate 2/12 AUC=0.902
2026-06-12 18:52:04,200 | INFO | full_aoi_lt6_rf_Hvala RF candidate 3/12 AUC=0.900
2026-06-12 18:52:14,795 | INFO | full_aoi_lt6_rf_Hvala RF candidate 4/12 AUC=0.898
2026-06-12 18:52:22,845 | INFO | full_aoi_lt6_rf_Hvala RF candidate 5/12 AUC=0.888
2026-06-12 18:52:31,230 | INFO | full_aoi_lt6_rf_Hvala RF candidate 6/12 AUC=0.887
2026-06-12 18:52:44,739 | INFO | full_aoi_lt6_rf_Hvala RF candidate 7/12 AUC=0.881
2026-06-12 18:52:57,990 | INFO | full_aoi_lt6_rf_Hvala RF candidate 8/12 AUC=0.891
2026-06-12 18:53:04,465 | INFO | full_aoi_lt6_rf_Hvala RF candidate 9/12 AUC=0.896
2026-06-12 18:53:14,928 | INFO | full_aoi_lt6_rf_Hvala RF candidate 10/12 AUC=0.898
2026-06-12 18:53:20,220 | INFO | full_aoi_lt6_rf_Hvala RF candidate 11/12 AUC=0.886
2026-06-12 18:53:30,005 | INFO | full_aoi_lt6_rf_Hvala RF candidate 12/12 AUC=0.898
2

,training_domain,cub_class,candidate,algorithm,evaluation_scenario_local,auc_roc_mean_local,auc_roc_std_local,auc_roc_min_local,auc_roc_max_local,auc_pr_mean_local,...,tss_mean_full,tss_std_full,tss_min_full,tss_max_full,robust_floor,mean_auc,mean_boyce,mean_brier,management_ready,reliability_status
0,full_aoi,6to12,rf,rf,local_test,0.736431,0.103992,0.646609,0.875657,0.655821,...,0.709531,0.126153,0.590447,0.87550,0.646609,0.813988,-0.112310,0.325313,False,exploratory
1,local_domain,lt6,rf,rf,local_test,0.760794,0.117924,0.630476,0.909908,0.717162,...,0.646137,0.150268,0.459387,0.82715,0.630476,0.817294,0.135941,0.344641,True,management_ready


,training_domain,cub_class,candidate,algorithm,evaluation_scenario,auc_roc_mean,auc_roc_std,auc_roc_min,auc_roc_max,auc_pr_mean,...,log_loss_min,log_loss_max,brier_mean,brier_std,brier_min,brier_max,tss_mean,tss_std,tss_min,tss_max
6,full_aoi,6to12,rf,rf,full_test,0.891544,0.056208,0.828669,0.958526,0.824526,...,0.559712,1.684996,0.317884,0.100047,0.215025,0.455193,0.709531,0.126153,0.590447,0.875500
4,full_aoi,6to12,blend_rf75_xgb25,blend,full_test,0.886325,0.061600,0.812019,0.956295,0.814488,...,0.552856,1.718275,0.325169,0.102832,0.211746,0.460029,0.707124,0.132837,0.571317,0.879500
2,full_aoi,6to12,blend_rf50_xgb50,blend,full_test,0.881116,0.067275,0.795685,0.953763,0.805549,...,0.550896,1.761015,0.333653,0.107214,0.209731,0.465214,0.702561,0.141393,0.547195,0.880500
0,full_aoi,6to12,blend_rf25_xgb75,blend,full_test,0.875334,0.074670,0.776307,0.951250,0.797642,...,0.554524,1.815017,0.343336,0.113290,0.208979,0.470748,0.698012,0.153867,0.518305,0.885500
8,full_aoi,6to12,xgb,xgb,full_test,0.867106,0.085638,0.749119,0.948583,0.788645,...,0.566043,1.886105,0.354220,0.121092,0.209491,0.476632,0.678869,0.165205,0.466569,0.861859
26,local_domain,6to12,rf,rf,full_test,0.810040,0.143720,0.613911,0.934103,0.753704,...,0.982604,1.649104,0.395760,0.041387,0.353852,0.452906,0.564288,0.242489,0.250195,0.805038
24,local_domain,6to12,blend_rf75_xgb25,blend,full_test,0.807082,0.146583,0.605911,0.932878,0.747371,...,1.110026,1.411126,0.405246,0.015860,0.382262,0.418422,0.563196,0.244288,0.244825,0.804538
22,local_domain,6to12,blend_rf50_xgb50,blend,full_test,0.803786,0.149868,0.596835,0.931468,0.740772,...,1.085804,1.595955,0.416855,0.022008,0.387677,0.436642,0.558823,0.248523,0.232333,0.801538
20,local_domain,6to12,blend_rf25_xgb75,blend,full_test,0.800105,0.153666,0.586365,0.929564,0.734403,...,0.977073,1.907028,0.430587,0.047267,0.360673,0.463620,0.552811,0.253056,0.219110,0.796038
28,local_domain,6to12,xgb,xgb,full_test,0.773831,0.146407,0.573711,0.892571,0.707095,...,0.900803,3.595205,0.446441,0.073019,0.337409,0.492604,0.494295,0.233488,0.208350,0.688974


auc_roc               boyce               brier          
               mean       std      mean       std      mean       std
cub_class                                                            
6to12      0.976465  0.002612  0.993939  0.006388  0.096815  0.007354
lt6        0.929164  0.007640  0.978182  0.024775  0.128439  0.005105

,cub_class,method,scenario,raw_brier,calibrated_brier
0,6to12,isotonic,full_test,0.091486,0.193816
1,lt6,isotonic,local_test,0.067217,0.219711


# 3. Final spatial products and independent audit

The selected configuration for each cub class is refitted on all treated GPS data using daily-block bootstrap replicates. Its mean is calibrated from weighted out-of-fold predictions. If a local-background model wins, full-Pyrenees predictions outside the observed multivariate environmental range are masked in the official suitability product and retained only in diagnostic rasters.

Two threshold sets are generated. Paper-compatible thresholds apply the exact CARTOBIO recurrence to the final raw map. Official conservative thresholds apply the same recurrence to calibrated out-of-fold presences and the calibrated final map. Uncertainty is split into daily bootstrap, held-out individual, algorithm, and training-domain components, then summarized as a relative index. OBS records are used only here as an independent audit. Expected output: exactly two official suitability rasters, two official zone rasters, EPSG:4326 figures, novelty/MESS diagnostics, readable SHAP plots, permutation importance, and OBS validation.

In [30]:
product_outputs = workflow.generate_final_products()
display(product_outputs["thresholds"])
display(product_outputs["zone_areas"])
display(product_outputs["importance"].sort_values("mean", ascending=False).head(20) if not product_outputs["importance"].empty else product_outputs["importance"])
display(product_outputs["obs_validation"])


2026-06-12 20:50:30,882 | INFO | Workflow complete with 93 run-owned output files.


,cub_class,threshold_set,adequate,good,optimal
0,6to12,paper_compatible,0.457817,0.575291,0.659881
1,6to12,conservative_oof_calibrated,0.334153,0.678430,0.740674
2,lt6,paper_compatible,0.323890,0.471734,0.590164
3,lt6,conservative_oof_calibrated,0.348748,0.501671,0.765303


,cub_class,threshold_set,zone,area_type,area_km2
0,6to12,conservative_oof_calibrated,adequate,exclusive,6437.64
1,6to12,conservative_oof_calibrated,adequate,cumulative,17945.24
2,6to12,conservative_oof_calibrated,good,exclusive,8723.71
3,6to12,conservative_oof_calibrated,good,cumulative,11507.60
4,6to12,conservative_oof_calibrated,optimal,exclusive,2783.89
5,6to12,conservative_oof_calibrated,optimal,cumulative,2783.89
6,6to12,paper_compatible,adequate,exclusive,574.85
7,6to12,paper_compatible,adequate,cumulative,869.97
8,6to12,paper_compatible,good,exclusive,203.67
9,6to12,paper_compatible,good,cumulative,295.12


,index,cub_class,feature,mean,std
40,16,lt6,human_logdistance_tracks,0.064337,0.029858
21,21,6to12,topography_ruggedness,0.055465,0.014610
20,20,6to12,topography_dem_elevation,0.052862,0.006375
45,21,lt6,topography_ruggedness,0.042709,0.016278
44,20,lt6,topography_dem_elevation,0.034753,0.034342
5,5,6to12,habitat_biomass_mean,0.033987,0.028847
15,15,6to12,human_logdistance_towns,0.022355,0.010730
14,14,6to12,human_logdistance_secondary_roads_transport_se...,0.020953,0.007040
29,5,lt6,habitat_biomass_mean,0.017935,0.011043
24,0,lt6,climate_mean_temperature,0.015792,0.018104


,cub_class,training_domain,candidate,n_obs_presence,n_background,mean_obs_suitability,mean_background_suitability,mannwhitney_p_greater,auc_roc,auc_pr,boyce,log_loss,brier,tss,threshold_youden
0,6to12,full_aoi,rf,901,901,0.360583,0.060364,1.026974e-208,0.919123,0.894708,0.903030,0.688378,0.236904,0.702553,0.058784
1,lt6,local_domain,rf,266,266,0.449445,0.075900,3.188173e-66,0.929674,0.921749,0.905779,0.562899,0.194092,0.729323,0.146073


# 4. Interpretation rules

Use LOIO summaries, especially worst-bear local performance, as the primary evidence of transferability. Treat random 75/25 metrics as optimistic paper-comparison values. Continuous suitability is the principal scientific product; categorical zones depend on the chosen threshold method. A model labelled `exploratory` failed the non-negative local Boyce guardrail and should not be used directly for management decisions. Relative uncertainty is a disagreement index, not a confidence interval or probability of error.